# Write and run a Method

Decision Layer analyzes semantic-layer metrics through registered Methods. The Web and AI use these Methods and record inputs and results in a Run.

This notebook follows **write a function → register locally → supply inputs → inspect results**.

1. Write and edit a grouped query.
2. Combine several queries in one Method.
3. Connect DoWhy calculations and analyze binary and continuous treatments.
4. Run the same functions against a real Cube connection.

Start with small prepared data; a server or AI connection is not needed for the practice examples.


## 1. Set up

Select the repository’s `.venv` kernel and run the cell below. It prepares the example code and table display.


In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

repository = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src/decision_layer").is_dir()
)
sys.path.insert(0, str(repository))

from decision_layer.dev import MethodSession
from decision_layer.methods import (
    Dimension, Entity, Estimate, Filter, Measure, Refused, Scope, Table, method,
)
import examples.methods.statistics.teaching_data as example_data


def show_table(data):
    if isinstance(data, dict):
        data = [{"field": key, "value": value} for key, value in data.items()]
    display(pd.DataFrame(data))

## Start with an ordinary Python function

This route does not require `async`, `ctx`, or `Measure` on your calculation. `rank_values(frame, descending=True)` in [native_calculation.py](../../../examples/methods/authoring/native_calculation.py) uses ordinary pandas and preserves equal ranks for ties.

Call it directly first, then register its input mapping **without changing the function**. The small in-memory table is a calculation unit test, not a local CSV/file input feature.


In [ ]:
from examples.methods.authoring.native_calculation import rank_values

original_frame = pd.DataFrame({"value": [30, 20, 20], "group": ["A", "B", "C"]})
standalone_ranking = rank_values(original_frame)
assert standalone_ranking["rank"].tolist() == [1, 2, 2]
show_table(standalone_ranking.to_dict(orient="records"))


### Register the input mapping and execution contract

`value → metric` and `group → group` map function columns to runtime binding roles. Bind `metric` to a discovered metric reference. Do not define revenue SQL here.

This route supports one aggregate or entity table, a name/version, `Table`/`Estimate` output, and typed scalar options. Built-in checks cover row count, required values, finite measures and duplicate grain. Register additional executable checks with `validators=[...]` and output conversion with `adapt=...`. These checks do not establish source completeness, official metric equivalence or causal assumptions.


In [ ]:
from decision_layer.methods import InputTable, calculation

native_rank = calculation(
    rank_values,
    name="workshop.native_rank",
    version="1.0.0",
    table=InputTable(
        measures={"value": "metric"},
        dimensions={"group": "group"},
        format="dataframe",
    ),
    requires_period=True,
)


### Execute the same calculation and inspect evidence

This is a strict query/response fixture test, not a Cube network integration test. `CalculationMethod` uses the existing Registry, extraction bounds and Run contracts. Use the existing `@method` route below when composing several queries, periods or filters directly.

Notebook functions work in trusted local trials. Server workers require importable module-level functions. Worker deadlines and environment stripping are not a filesystem/network sandbox for arbitrary Python.


In [ ]:
from examples.methods.authoring.native_calculation import fixture_session
from examples.methods.first_method.fixtures import METRIC, DIMENSION, DATE, DATES

native_session = await fixture_session()
native_session.register(native_rank, replace=True)
native_trial = await native_session.run(
    native_rank, metric=METRIC, group=DIMENSION, scope=Scope(DATES, DATE),
)
assert native_trial.result.primary.data == standalone_ranking.to_dict(orient="records")
assert native_trial.result.status == "success"
show_table(native_trial.result.primary.data)
show_table([item.model_dump() for item in native_trial.result.validation])
assert len(native_trial.queries) == 1
native_session.close()


## 2. Write a grouped-query Method

Group metrics by one or more dimensions and return the top `limit` combinations by the first metric. Use a list even when selecting one metric or dimension.


In [ ]:
@method("workshop.by_group", version="1.0.0")
async def my_by_group(ctx, *, selected_measures: list[Measure],
                      group_by: list[Dimension], limit: int = 3) -> Table:
    """Return the largest grouped measure values, ordered by the first measure. No period comparison or causal claim."""
    data = await ctx.aggregate(
        measures={f"value_{i}": ref for i, ref in enumerate(selected_measures, 1)},
        dimensions={f"group_{i}": ref for i, ref in enumerate(group_by, 1)},
        order=[(selected_measures[0], "desc")],
        limit=limit,
    )
    return Table.from_dataset(data, require_non_empty=True)

`@method(...)` declares the call name and version. `workshop.by_group` is a practice name.

The `"""..."""` immediately below the function signature is its docstring. It becomes the registered description shown to the Web and AI.


`ctx` is the data-query object received by the function. Specify the query in `ctx.aggregate(...)`.

- `dimensions`: grouping dimensions and output column names, here `group_1`, `group_2`.
- `measures`: metrics and output column names, here `value_1`, `value_2`.
- `order`, `limit`: sort by the first metric descending and limit the returned rows.

`Table.from_dataset(data)` returns the query as a table. `require_non_empty=True` refuses an empty result.


### Register and run

Test the function in the notebook before installing it on a server.

- `example_session("by_group")`: create a local execution environment for this example’s data.
- `register(my_by_group)`: register the function in this environment.
- `run(...)`: execute with metrics, grouping dimensions and a period.

`cube://practice/observations/value` refers to the `value` metric in the `observations` model of the practice connection. Real data use paths from the connected semantic layer.


In [ ]:
group_session = await example_data.example_session("by_group")
group_session.register(my_by_group)

trial = await group_session.run(
    my_by_group,
    selected_measures=["cube://practice/observations/value"],
    group_by=["cube://practice/observations/group"],
    scope=Scope(
        ("2026-02-01", "2026-02-28"),
        "cube://practice/observations/date",
    ),
    limit=3,
)

show_table(trial.result.primary.data)

group_session.close()

`run(...)` builds `ctx` from the connection and the period and filters in `scope`, then passes it to the function. `selected_measures`, `group_by` and `limit` become the corresponding function arguments.

This example returns `30, 20, 10`. Re-running the execution cell creates a fresh practice environment.


### Inspect the registered description

The cell below shows the Method name, docstring description and required inputs that AI can inspect, using the local environment.


In [ ]:
method_choices = group_session.find_methods()
show_table([
    {"method": item["manifest"]["name"],
     "description": item["manifest"]["description"],
     "inputs": ", ".join(item["required_inputs"])}
    for item in method_choices["candidates"]
])


### From an AI request to Method execution

Example: “Show the top 3 departments by sales amount for store 364 in February 2001.”

If `workshop.by_group` is registered on the server, AI inspects and executes it through these MCP tools.

| MCP tool | Information or action | Use |
| --- | --- | --- |
| `search_semantic` | Names and paths of metrics, dimensions, dates and filter objects | Find sales amount, department, transaction date and store |
| `find_methods` | Candidate descriptions and input requirements | Narrow down suitable candidates |
| `find_recipes` | Reusable procedures and goal coverage | Use a suitable procedure or choose a Method directly |
| `describe_method` | Selected Method’s `description`, `roles`, `parameters` | Read its docstring, semantic inputs and `limit` default |
| `start_analysis` | Create a Run with the original question, period and shared filters | Record February 2001 and store 364; use `start_run` for a Recipe |
| `run_step` | Execute a Method by name and inputs | Supply the inputs to `workshop.by_group` |
| `complete_run` | Save the evidence-linked answer and execution record | Return results and a Run link |

The inputs returned by `describe_method("workshop.by_group")` come from the function above.

| Function input | Information from `describe_method` | Value for the request |
| --- | --- | --- |
| `selected_measures` | Required list of metric paths | `["cube://journey/transaction/receipts"]` |
| `group_by` | Required list of dimension paths | `["cube://journey/product/department"]` |
| `limit` | Integer, default `3` | `3` |

These paths refer to semantic objects discovered with `search_semantic`. Practice cells use the prepared `practice` model instead of `journey`.

AI fills `run_step`’s `measures={"selected_measures": [...]}`, `dimensions={"group_by": [...]}` and `params={"limit": 3}`. The period and store filter apply through the Run scope. Ambiguous metrics, periods or multiple possible analyses require clarification.

In this notebook, supply the same inputs directly to `run(...)`. Notebook `register(...)` is local only. Web and MCP use a function installed and registered in reviewed server startup code.


### Inspect the query record

If the result differs from expectations, first check the requested period, filters and execution status.


In [ ]:
show_table([
    {
        "query": index,
        "label": attempt.label,
        "status": attempt.status,
        "period": attempt.spec.time.date_range,
        "filters": [item.model_dump() for item in attempt.spec.filters],
    }
    for index, attempt in enumerate(trial.attempts, start=1)
])

### Use multiple measures and dimensions

Pass two measures and two dimensions to the same Method. It queries both measures for each `group_1` and `group_2` combination, sorted by the first measure.

In [ ]:
multi_session = await example_data.example_session("multi_group")
multi_session.register(my_by_group)

multi_trial = await multi_session.run(
    my_by_group,
    selected_measures=[
        "cube://practice/observations/value",
        "cube://practice/observations/before_value",
    ],
    group_by=[
        "cube://practice/observations/group",
        "cube://practice/observations/subject",
    ],
    scope=Scope(("2026-02-01", "2026-02-28"), "cube://practice/observations/date"),
    limit=3,
)

assert multi_trial.result.status == "success"
assert multi_trial.result.primary.data[0] == {
    "value_1": 30, "value_2": 25, "group_1": "A", "group_2": "X",
}
show_table(multi_trial.result.primary.data)
multi_session.close()

### Write a monthly query

Use `time_granularity="month"` for monthly values. Return the same `Table`, with dates in the first column.


In [ ]:
@method("workshop.monthly", version="1.0.0")
async def my_monthly(ctx, *, selected_measures: list[Measure]) -> Table:
    """Return monthly measure values within the requested period. No change significance test."""
    data = await ctx.aggregate(
        measures={f"value_{i}": ref for i, ref in enumerate(selected_measures, 1)},
        time_granularity="month",
    )
    return Table.from_dataset(data, require_non_empty=True)

In [ ]:
monthly_session = await example_data.example_session("trend")
monthly_session.register(my_monthly)

monthly_trial = await monthly_session.run(
    my_monthly,
    selected_measures=["cube://practice/observations/value"],
    scope=Scope(
        ("2026-01-01", "2026-02-28"),
        "cube://practice/observations/date",
    ),
)

show_table(monthly_trial.result.primary.data)
monthly_session.close()

,time,value_1
0,2026-01-01,20
1,2026-02-01,30


## 3. Edit the code: add a rank

Add `rank` to the grouped query. Keep the name and use version `1.1.0` to identify the edited implementation.


In [ ]:
@method("workshop.by_group", version="1.1.0")
async def my_ranked_by_group(ctx, *, selected_measures: list[Measure],
                             group_by: list[Dimension], limit: int = 3) -> Table:
    """Order by the first measure descending and add a group rank."""
    data = await ctx.aggregate(
        measures={f"value_{i}": ref for i, ref in enumerate(selected_measures, 1)},
        dimensions={f"group_{i}": ref for i, ref in enumerate(group_by, 1)},
        order=[(selected_measures[0], "desc")],
        limit=limit,
    )
    rows = data.records()
    for rank, row in enumerate(rows, start=1):
        row["rank"] = rank
    return Table(rows)

### Run the edited function

Register only the edited Method in a new practice session and run it. The result adds ranks `1, 2, 3`.

In [ ]:
ranked_session = await example_data.example_session("by_group")
ranked_session.register(my_ranked_by_group)

edited_trial = await ranked_session.run(
    my_ranked_by_group,
    selected_measures=["cube://practice/observations/value"],
    group_by=["cube://practice/observations/group"],
    scope=Scope(
        ("2026-02-01", "2026-02-28"),
        "cube://practice/observations/date",
    ),
    limit=3,
)

assert edited_trial.result.status == "success"
assert [row["rank"] for row in edited_trial.result.primary.data] == [1, 2, 3]
assert [row["value_1"] for row in edited_trial.result.primary.data] == [30, 20, 10]
show_table(edited_trial.result.primary.data)

ranked_session.close()

## 4. Combine multiple queries in one Method

Make three queries in the same function. Extract each value, calculate differences and return one table.

| Query | Condition |
| --- | --- |
| `subject` | Subject A in group X |
| `peers` | Group X excluding A |
| `overall` | All units excluding A |

All queries use the same `ctx`, so they share the period and common filters. The function rejects subject A in shared filters because this would also restrict the comparison groups to A.


In [ ]:
@method(
    "workshop.peer",
    version="1.0.0",
)
async def my_peer(
    ctx,
    *,
    measure: Measure,
    subject: Dimension,
    category: Dimension,
    subject_value: str,
    category_value: str,
) -> Table:
    """Compare a subject with explicitly selected peer groups. Differences do not establish causality."""
    if any(item.member == subject for item in ctx.scope.filters):
        raise Refused("Keep the subject out of shared filters")

    same_category = Filter(
        member=category,
        operator="equals",
        values=[category_value],
    )
    selected = Filter(member=subject, operator="equals", values=[subject_value])
    excluded = Filter(member=subject, operator="notEquals", values=[subject_value])

    subject_data = await ctx.aggregate(
        measures={"value": measure},
        filters=[same_category, selected],
        label="subject",
    )
    peer_data = await ctx.aggregate(
        measures={"value": measure},
        filters=[same_category, excluded],
        label="peers",
    )
    overall_data = await ctx.aggregate(
        measures={"value": measure},
        filters=[excluded],
        label="overall",
    )

    subject_result = subject_data.scalar("value")
    peer_result = peer_data.scalar("value")
    overall_result = overall_data.scalar("value")

    return Table(
        [
            {
                "comparison": "peers",
                "subject": subject_result,
                "benchmark": peer_result,
                "difference": subject_result - peer_result,
            },
            {
                "comparison": "overall",
                "subject": subject_result,
                "benchmark": overall_result,
                "difference": subject_result - overall_result,
            },
        ],
        warnings=["Descriptive comparison; no causal claim or significance test."],
    )

### Register and run

Select subject A and group X. The prepared responses are `12, 8, 6`, giving differences of `4, 6`.

In [ ]:
peer_session = await example_data.example_session("peer")
peer_session.register(my_peer)

peer_trial = await peer_session.run(
    my_peer,
    measure="cube://practice/observations/rate",
    subject="cube://practice/observations/subject",
    category="cube://practice/observations/group",
    scope=Scope(
        ("2026-02-01", "2026-02-28"),
        "cube://practice/observations/date",
    ),
    subject_value="A",
    category_value="X",
)

assert peer_trial.result.status == "success"
assert [row["difference"] for row in peer_trial.result.primary.data] == [4, 6]
assert len(peer_trial.attempts) == 3
show_table(peer_trial.result.primary.data)

peer_session.close()

### Inspect the three queries

Compare their labels and filters. As an editing exercise, add `ratio` to the result.

In [ ]:
show_table([
    {
        "query": index,
        "label": attempt.label,
        "status": attempt.status,
        "filters": [item.model_dump() for item in attempt.spec.filters],
    }
    for index, attempt in enumerate(peer_trial.attempts, start=1)
])

## 5. Connect a Python calculation to a Method

Retrieve per-unit observations for DoWhy regression and return the calculation as a Method result.

| Input | Value in this example |
| --- | --- |
| `outcome` | Outcome metric |
| `treatment` | Treatment encoded as 0 or 1 |
| `confounders` | Adjustment variables measured before treatment |
| `unit` | An observation entity, such as a customer |

This authoring example uses binary treatment. Later, run the default Method with continuous treatment. Identification assumptions remain unverified and no interval is reported.


### Prepare the calculation

Import the ordinary Python calculation in [binary_regression.py](../../../examples/methods/statistics/binary_regression.py). It checks missing values, duplicate units, sample sizes and observed support, then directly calls DoWhy’s `model.estimate_effect(..., method_name="backdoor.linear_regression")`.

Open that file to review or edit the implementation. Here, connect the same function to governed data retrieval.


In [ ]:
from examples.methods.statistics.binary_regression import estimate_binary_regression


### Connect retrieval and calculation

- `ctx.entities(...)`: retrieve selected metrics and variables per unit.
- `ctx.compute(...)`: pass the resulting DataFrame to the calculation function.
- `Estimate(...)`: return the estimate and analysis details.


In [ ]:
@method(
    "workshop.binary_regression",
    version="1.0.0",
    execution="dataframe",
    requires_packages={
        "dowhy": "==0.13",
        "numpy": ">=1.26,<2",
        "pandas": ">=2.2,<3",
        "scikit-learn": ">=1.2,<1.7",
    },
)
async def my_binary_regression(
    ctx,
    *,
    outcome: Measure,
    treatment: Dimension,
    confounders: list[Dimension | Measure],
    unit: Entity,
) -> Estimate:
    # Extract governed entity data, then run ordinary Python calculation code.
    observations = await ctx.entities(
        entity=unit,
        measures={"outcome": outcome},
        dimensions={
            "treatment": treatment,
            "confounders": confounders,
        },
    )
    payload = await ctx.compute(
        estimate_binary_regression,
        observations.frame,
        observations.columns("confounders"),
    )

    return Estimate(
        payload["value"],
        details={
            **payload,
            "assumptions_status": "not_verified",
        },
        warnings=[
            "Unverified assumptions and placebo diagnostics do not establish "
            "causality; no interval is reported."
        ],
    )

### Register and run

Connect the outcome, treatment, adjustment variables and observation unit. The example data follows `Y = 2*T + 3*W + noise`, with an expected value near `2`.

In [ ]:
regression_session = await example_data.example_session("dowhy")
regression_session.register(my_binary_regression)

dowhy_trial = await regression_session.run(
    my_binary_regression,
    outcome="cube://practice/observations/value",
    treatment="cube://practice/observations/treated",
    confounders=["cube://practice/observations/baseline"],
    unit="cube://practice/observations/id",
    scope=Scope(
        ("2026-02-01", "2026-02-28"),
        "cube://practice/observations/date",
    ),
)

assert dowhy_trial.result.status == "success"
assert abs(dowhy_trial.result.primary.data["value"] - 2) < 0.15
show_table(dowhy_trial.result.primary.data)

regression_session.close()


### Use it in ordinary Python

Pass a DataFrame directly to the same calculation function and compare its value with the Method result.

In [ ]:
frame = example_data.binary_regression_frame()
standalone_result = estimate_binary_regression(frame, ["baseline"])

assert abs(standalone_result["value"] - 2) < 0.15
assert standalone_result["value"] == dowhy_trial.result.primary.data["value"]
show_table(standalone_result)


### Inspect a refusal

Use practice data containing a duplicate observation unit. The Method refuses estimation and returns a reason.

In [ ]:
refusal_session = await example_data.duplicate_unit_session()
refusal_session.register(my_binary_regression)

refused_trial = await refusal_session.run(
    my_binary_regression,
    outcome="cube://practice/observations/value",
    treatment="cube://practice/observations/treated",
    confounders=["cube://practice/observations/baseline"],
    unit="cube://practice/observations/id",
    scope=Scope(
        ("2026-02-01", "2026-02-28"),
        "cube://practice/observations/date",
    ),
)

assert refused_trial.result.status == "refused"
show_table([
    {"status": refused_trial.result.status, "reason": warning}
    for warning in refused_trial.result.warnings
])
refusal_session.close()


### Run the two default DoWhy analyses

- `stats.binary_matching`: match controls on pre-treatment covariates and estimate the treated population contrast (ATT).
- `stats.regression_adjustment`: use DoWhy OLS to estimate the adjusted change per treatment unit, with optional explicit contrasts.

The code runs registered Methods with the same four inputs. The synthetic data have a known contrast of 2. Real identification assumptions are unverified and no uncertainty interval is reported. See `dowhy_basics.py` for the entity queries and execution code.


In [ ]:
from examples.methods.statistics.dowhy_basics import demonstrate

show_table(await demonstrate())


### Continuous treatment: supply the variable

Synthetic discount rates range from 0.05 to 0.30, with outcome `100 + 50 × discount + 3 × baseline`.

- The default call returns **50** per stored treatment unit (1.0), equivalent to **0.5** per percentage point (0.01).
- An explicit 10% to 20% contrast returns **5**.
- Check treatment units in the semantic definition. The Method does not convert percentages automatically.

The per-unit result is the constant slope of a linear model. It does not establish effects of actual interventions outside observed values.


In [ ]:
from examples.methods.statistics import continuous_regression


In [ ]:
continuous_session = await continuous_regression.example_session()

continuous_trial = await continuous_session.run(
    "stats.regression_adjustment",
    outcome="cube://practice/observations/value",
    treatment="cube://practice/observations/rate",
    confounders=["cube://practice/observations/baseline"],
    unit="cube://practice/observations/id",
    scope=Scope(("2026-02-01", "2026-02-28"), "cube://practice/observations/date"),
)
show_table([{
    "result": "per stored treatment unit",
    "value": continuous_trial.result.primary.data["value"],
}])
continuous_session.close()


In [ ]:
continuous_session = await continuous_regression.example_session()

contrast_trial = await continuous_session.run(
    "stats.regression_adjustment",
    outcome="cube://practice/observations/value",
    treatment="cube://practice/observations/rate",
    confounders=["cube://practice/observations/baseline"],
    unit="cube://practice/observations/id",
    scope=Scope(("2026-02-01", "2026-02-28"), "cube://practice/observations/date"),
    control_value=0.10,
    treatment_value=0.20,
)
show_table([{
    "result": "0.10 to 0.20",
    "value": contrast_trial.result.primary.data["value"],
}])
continuous_session.close()


## 6. Run against real Cube data

Query the connected Cube instead of prepared responses. Set `RUN_LIVE=True` in the connection cell and run it. Default connection values are for the local Journey sample.

This is also local Python execution; it does not register code on the server or change the Web catalog.


In [ ]:
import os
from decision_layer.semantic.credentials import ServiceCredentials

RUN_LIVE = False
live = None

if RUN_LIVE:
    live = await MethodSession.cube(
        os.environ.get("DL_JOURNEY_CUBE_URL", "http://127.0.0.1:4000/cubejs-api/v1"),
        ServiceCredentials(
            os.environ.get(
                "DL_JOURNEY_CUBE_SECRET",
                "local-example-secret-change-me-0123456789",
            ),
            (),
        ),
        instance="journey",
    )

show_table([{
    "status": "connected" if live is not None else "skipped",
    "reason": "Cube connected" if live is not None else "Set RUN_LIVE = True in this connection cell",
}])

### Find the connected metrics and dimensions

Check the real catalog for the paths of `receipts` and `department`, then supply them to the execution code below.


In [ ]:
if live is not None:
    objects = [
        *live.catalog.search(kind="measure", text="receipts"),
        *live.catalog.search(kind="dimension", text="department"),
    ]
    show_table([
        {"title": obj.title, "kind": obj.kind, "path": obj.ref}
        for obj in objects
    ])
else:
    show_table([{"status": "skipped", "reason": "Enable the Cube connection above"}])

### Query by group

Query retailer receipts by department. Try changing `limit` or the period.

In [ ]:
if live is not None:
    live.register(my_by_group, replace=True)
    live_group_trial = await live.run(
        my_by_group,
        selected_measures=["cube://journey/transaction/receipts"],
        group_by=["cube://journey/product/department"],
        scope=Scope(
            ("2001-01-01", "2001-06-30"),
            "cube://journey/transaction/analysis_date",
        ),
        limit=3,
    )
    show_table(live_group_trial.result.primary.data)
else:
    show_table([{"status": "skipped", "reason": "Enable the Cube connection above"}])

### Use the same Cube connection for the native calculation

Run `native_rank` against the Journey references inspected above. Source aggregation supplies the complete bounded set of groups; extraction exceeding the row bound is refused. This ranks receipts and does not establish causality.


In [ ]:
if live is not None:
    live.register(native_rank, replace=True)
    live_native_trial = await live.run(
        native_rank,
        metric="cube://journey/transaction/receipts",
        group="cube://journey/product/department",
        scope=Scope(("2001-01-01", "2001-06-30"), "cube://journey/transaction/analysis_date"),
    )
    show_table(live_native_trial.result.primary.data if live_native_trial.result.primary else [])
    show_table([item.model_dump() for item in live_native_trial.result.validation])
else:
    show_table([{"status": "skipped", "reason": "Enable the Cube connection above"}])


### Compare populations

Compare store 364 in GROCERY with its peers and the overall population. Try other stores or departments present in the data.

In [ ]:
if live is not None:
    live.register(my_peer, replace=True)
    live_peer_trial = await live.run(
        my_peer,
        measure="cube://journey/transaction/coupon_line_rate",
        subject="cube://journey/transaction/store",
        category="cube://journey/product/department",
        scope=Scope(
            ("2001-01-01", "2001-06-30"),
            "cube://journey/transaction/analysis_date",
        ),
        subject_value="364",
        category_value="GROCERY",
    )
    show_table(live_peer_trial.result.primary.data)
    show_table([
        {"query": index, "label": attempt.label, "status": attempt.status}
        for index, attempt in enumerate(live_peer_trial.attempts, start=1)
    ])
else:
    show_table([{"status": "skipped", "reason": "Enable the Cube connection above"}])

### Finish

Close the local connection. To offer your Method through Web or MCP, save the function in a Python file, register it at server startup and deploy it.


In [ ]:
if live is not None:
    live.close()
    live = None

show_table([{"status": "closed"}])